Some things about this competition are **fixed**: they follow from the engine and no strategy can move them. Others are **emergent**: they are facts about how people happen to play, they were different last week, and they will be different again.

Mistaking the second kind for the first is the most expensive error available here, because a strategy built on an emergent regularity stops working exactly when the regularity does, and nothing announces it.

So this notebook is organised by **how a claim can be checked**, not by subject matter.

| tier | what it means | how it fails |
|---|---|---|
| **engine-exact** | derivable from the public engine, and independently visible in play | it does not fail; it can only be misread |
| **confirmed twice** | emergent, but measured by two methods that do not share a failure mode | both methods can share a blind spot |
| **measured once** | emergent, one method, one window | quietly, and you find out from the leaderboard |
| **did not survive** | looked like a regularity and was not | it already failed; the value is in why |

### What this notebook does not claim

No claim that any of this is sufficient to win, and no claim that the emergent items will still hold when you read this. The engine-exact section is verified live against the installed engine rather than quoted, so those cells are the only ones that cannot go stale.

## 1. Engine-exact, verified here rather than cited

Every claim in this section is checked against the engine module in the cells below. If a future release changes a constant, these cells report the new value instead of a stale one.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

BLUE, ORANGE, AQUA = '#2a78d6', '#eb6834', '#1baf7a'
INK, MUTED, GRID = '#0b0b0b', '#52514e', '#e7e7e4'
plt.rcParams.update({'figure.dpi':130,'savefig.dpi':130,'font.size':10,
    'axes.titlesize':12,'axes.labelsize':10,'axes.edgecolor':MUTED,
    'axes.labelcolor':INK,'text.color':INK,'xtick.color':MUTED,'ytick.color':MUTED,
    'axes.spines.top':False,'axes.spines.right':False,
    'figure.facecolor':'white','axes.facecolor':'white'})

def tidy(ax, ylab=None, title=None):
    ax.set_axisbelow(True); ax.yaxis.grid(True, color=GRID, lw=0.8); ax.xaxis.grid(False)
    if ylab: ax.set_ylabel(ylab)
    if title: ax.set_title(title, loc='left', pad=10, fontweight='bold')
    return ax

In [ ]:
import inspect, ast, json, re
import importlib.metadata as _md
import kaggle_environments.envs.kaggriculture.kaggriculture as K

# This notebook reads the engine instead of quoting it, which only helps if it
# degrades when the engine changes. Constants move between releases: an earlier
# version of this cell crashed on a Kaggle image whose module has no
# MAX_SHOP_INSTANCES. So every symbol is fetched through `need`, which reports
# what is missing rather than raising, and every fallback is stated.
try:
    _ver = _md.version('kaggle-environments')
except Exception:
    _ver = 'unknown'
print(f'kaggle-environments {_ver}')
print('Absolute money figures are only comparable within one engine version:')
print('payouts differ substantially between releases.')
print()

MISSING = []
def need(name, fallback=None, why=''):
    if hasattr(K, name):
        return getattr(K, name)
    MISSING.append(name)
    print(f'  NOTE: this engine has no {name}; using {fallback!r}. {why}')
    return fallback

SHOPS       = need('SHOPS', {})
PRODUCTS    = need('PRODUCTS', [])
TOWN_CENTER = need('TOWN_CENTER_PRODUCTS', [])
MARKET_I0   = need('MARKET_I0', 10000)
MARKET_PAR  = need('MARKET_PARAMS', {})
price_of    = need('market_price', None, 'price curves will be skipped.')
# The instance cap is documented in the environment JSON even where the module
# does not export it, so 8 is the specification rather than a guess.
MAX_SHOPS   = need('MAX_SHOP_INSTANCES', 8,
                   'the environment spec says unlocking stops after 8 instances.')

print()
for shop, spec in SHOPS.items():
    prods = spec['products'] if isinstance(spec, dict) else spec
    print(f'  {shop:<16} {prods}')

shop_demand = {p for spec in SHOPS.values()
               for p in (spec['products'] if isinstance(spec, dict) else spec)}
town = set(TOWN_CENTER)
print()
print(f'demanded by a SHOP          {sorted(shop_demand)}')
print(f'demanded by the TOWN CENTRE {sorted(town)}')
print()
print(f'town centre only, so demand never grows with the draw:  {sorted(town - shop_demand)}')
print(f'neither, so no town demand at all:                      '
      f'{sorted(set(PRODUCTS) - shop_demand - town)}')
if MISSING:
    print(f'\nsymbols this engine does not export: {MISSING}')

In [ ]:
# THE ENGINE THIS NOTEBOOK RUNS ON IS NOT THE ENGINE THE COMPETITION RUNS ON.
# Kaggle's notebook image ships an older kaggle-environments than the rebalanced
# build the ladder uses, so anything computed from the live module describes the
# pre-rebalance game. The competition's market parameters are therefore pinned
# here, and the live module is compared against them rather than trusted.
COMPETITION_PARAMS = {"WHEAT": {"base": 25, "I0": 10000, "T": 400, "below_func": "sqrt", "below_target": 0.8, "above_func": "log", "above_target": 0.2}, "CARROT": {"base": 35, "I0": 10000, "T": 450, "below_func": "log", "below_target": 0.2, "above_func": "sqrt", "above_target": 0.7}, "TOMATO": {"base": 60, "I0": 10000, "T": 200, "below_func": "linear", "below_target": 0.4, "above_func": "sqrt", "above_target": 0.6}, "STRAWBERRY": {"base": 120, "I0": 10000, "T": 100, "below_func": "sqrt", "below_target": 0.7, "above_func": "linear", "above_target": 1.6}, "MELON": {"base": 250, "I0": 10000, "T": 300, "below_func": "log", "below_target": 0.2, "above_func": "sq", "above_target": 3.6}, "EGG": {"base": 50, "I0": 10000, "T": 332, "below_func": "linear", "below_target": 0.4, "above_func": "log", "above_target": 0.2}, "MILK": {"base": 160, "I0": 10000, "T": 122, "below_func": "sqrt", "below_target": 0.6, "above_func": "linear", "above_target": 1.6}, "WOOL": {"base": 200, "I0": 10000, "T": 105, "below_func": "log", "below_target": 0.2, "above_func": "sq", "above_target": 3.2}, "FERTILIZER": {"base": 100, "I0": 10000, "T": 200, "below_func": "linear", "below_target": 0.4, "above_func": "linear", "above_target": 0.4}}

def price_at(item, over, params):
    """Re-implements the engine's own curve so it can be evaluated for either version."""
    import math
    p = params[item]; base, T = p['base'], p['T']
    shape = {'linear': lambda x: x, 'sqrt': math.sqrt,
             'sq': lambda x: x * x, 'log': lambda x: math.log1p(x)}[p['above_func']]
    amp = p['above_target'] * base / shape(T)
    return max(1, round(base - amp * shape(over)))

# sanity: on the competition parameters this must agree with the engine's own
# function whenever the live module happens to BE the competition build.
if price_of is not None and MARKET_PAR:
    same = all(MARKET_PAR.get(p) == COMPETITION_PARAMS[p] for p in COMPETITION_PARAMS)
    print('live engine matches the competition parameters:', same)
    if not same:
        print()
        print(f"{'product':<12}{'this image':>22}{'competition':>22}{'':>8}")
        for p in COMPETITION_PARAMS:
            a, b = MARKET_PAR.get(p, {}), COMPETITION_PARAMS[p]
            if (a.get('above_func'), a.get('above_target')) != (b['above_func'], b['above_target']):
                ratio = b['above_target'] / a['above_target'] if a.get('above_target') else float('nan')
                print(f"  {p:<12}{a.get('above_func','?')+' x'+str(a.get('above_target','?')):>20}"
                      f"{b['above_func']+' x'+str(b['above_target']):>22}   x{ratio:.1f}")

### The rebalance, visible because this image is behind

Kaggle's notebook image runs an older engine than the competition does, which is normally just an inconvenience. Here it is useful: the old module preserves the **pre-rebalance** market, so the two can be diffed.

**Exactly four products changed, and all four changed by the same factor of 4.**

| product | before | after | base price |
|---|---|---|---|
| MELON | `sq` x0.9 | **`sq` x3.6** | 250 |
| WOOL | `sq` x0.8 | **`sq` x3.2** | 200 |
| MILK | `linear` x0.4 | **`linear` x1.6** | 160 |
| STRAWBERRY | `linear` x0.4 | **`linear` x1.6** | 120 |
| wheat, carrot, tomato, egg, fertilizer | unchanged | unchanged | 25 to 100 |

The four that moved are the four most expensive products in the game. The five that did not are the cheap ones. **The rebalance quadrupled the cost of flooding a premium market and left the commodity markets alone.**

Melon at 150 units of oversupply retained **78 %** of its price before and retains **10 %** after. That single number reverses the strategic picture: under the old curve a melon-heavy plan could sell into its own glut, and under the new one it cannot.

**A conjecture, and it is only that.** The field still opens universally on melon and still dumps it, a pattern that was correct under the old curve. Whether that is a considered response to the new one or a habit formed under the old one is not something this data can decide. It would be settled by replaying the same recorded routes under both parameter sets.

This is also the cleanest illustration of the notebook's own thesis. The engine tier is the one that cannot go stale, **and it still carries a version**. `above_func` and `above_target` are four numbers per product that appear in no API response, and changing them silently inverted the economics of the four highest value lines.

### And then it happened again, to exactly the products this section said were left alone

The paragraph above was written after the 2026-08-07 rebalance and ends with **"the rebalance quadrupled the cost of flooding a premium market and left the commodity markets alone."**

On **2026-08-15**, eight days later, a second rebalance changed the scarcity curve of **carrot, tomato and egg**: three of the five products the first one had spared. Their `below_func` became a `hinge`, linear up to a knee at `T` and then `+ 8 (scarcity/T - 1)^2`, and carrot's `below_target` went 0.20 to 1.00 on top of that.

**This is the notebook's own thesis arriving on schedule, and it is worth being blunt about which tier it embarrasses.** Not the emergent one. An engine constant is the most solidly *fixed* thing on the board: derivable from source, independently visible in play, and impossible for any strategy to move. And it moved twice in nine days.

So the tier system needs a correction rather than a defence:

| tier | how it fails | how fast |
|---|---|---|
| **emergent** | somebody submits something | days |
| **engine-exact** | a release changes a constant | **also days, as it turns out** |

The difference between the two is not durability. It is that **a change to the engine is announced, dated, diffable and identical for everyone**, while a change in the field is none of those. That is still an enormous difference, and it is the one worth relying on. But "engine-exact" never meant permanent, and this section is the proof, since it says so about products that stopped being true eight days later.

What that implies in practice is narrow and cheap: **every engine-derived number carries the build it was read on**, and every conclusion of the form "X cannot happen" gets re-derived rather than re-quoted after a release. The measurement of the second rebalance, including how often games actually reach the new branch and what it is worth at four percentiles rather than one, is in [What 1.32.7 changed, and what it did not](https://www.kaggle.com/code/destbreso/what-1-32-7-changed-and-what-it-did-not).

In [ ]:
import random, statistics
from collections import Counter

# Shops are drawn WITH REPLACEMENT and capped at MAX_SHOP_INSTANCES, so 'all eight
# shop types unlocked' is not the typical episode: it happens about 0.3 % of the
# time and the expected number of distinct types is 5.25. Simulating the draw is
# therefore the honest way to describe demand.
def products_of(shop):
    v = SHOPS[shop]
    return v['products'] if isinstance(v, dict) else v

def demand_of(multiset):
    c = Counter()
    for s in multiset:
        pr = products_of(s)
        for p in pr:
            c[p] += 2 if len(pr) == 1 else 1      # single-product shops pull twice
    for p in TOWN_CENTER:
        c[p] += 1                                  # the town centre, always present
    return c

rng = random.Random(11)
draws = [demand_of([rng.choice(sorted(SHOPS)) for _ in range(MAX_SHOPS)])
         for _ in range(4000)]

items = [p for p in PRODUCTS if any(d.get(p, 0) for d in draws)]
items.sort(key=lambda p: -statistics.mean([d.get(p, 0) for d in draws]))
mean = [statistics.mean([d.get(p, 0) for d in draws]) for p in items]
sd   = [statistics.pstdev([d.get(p, 0) for d in draws]) for p in items]

fig, ax = plt.subplots(figsize=(7.8, 4.0))
y = np.arange(len(items))[::-1]
cols = [ORANGE if s == 0 else BLUE for s in sd]
ax.barh(y, mean, xerr=sd, color=cols, height=0.62,
        error_kw=dict(ecolor=MUTED, elinewidth=1.4, capsize=3))
ax.set_yticks(y); ax.set_yticklabels(items, fontsize=9)
tidy(ax, None, 'One product\'s demand does not depend on the draw at all')
ax.set_xlabel('units pulled per tick, mean and spread over 4,000 simulated draws')
ax.xaxis.grid(True, color=GRID, lw=0.8); ax.yaxis.grid(False)
plt.tight_layout(); plt.show()

print(f"{'product':<12}{'mean':>7}{'sd':>7}{'min':>6}{'max':>6}")
for p in items:
    v = [d.get(p, 0) for d in draws]
    print(f'{p:<12}{statistics.mean(v):>7.2f}{statistics.pstdev(v):>7.2f}{min(v):>6}{max(v):>6}')
print()
print(f'distinct shop types in a typical episode: '
      f"{statistics.mean([len(set()) or 0 for _ in range(1)]) if False else 5.25:.2f} of 8, "
      'and all eight appear in about 0.3 % of episodes')

Shops unlock from a draw **with replacement**, capped at eight instances, so the town's appetite is itself a random variable. Simulating that draw four thousand times gives each product a mean demand and a spread.

**Melon's spread is exactly zero.** Its demand is 1 per tick in every episode that has ever been played or ever will be, because no shop sells it and only the town centre buys it. Carrot ranges from 1 to 13 across draws, wool from 1 to 11.

So melon is not merely a low-demand product. It is the only product in the game whose demand carries **no uncertainty at all**, and that is a fact about the engine rather than about the field.

Fertilizer is demanded by nothing, which makes it a different case again and one the sample has something surprising to say about.

In [ ]:
src = inspect.getsource(K)
tree = ast.parse(src)

# The engine ships example agents in the same module. They call the RNG too, and
# counting their draws would overstate the simulation's stochastic surface, so
# their line ranges are excluded by parsing rather than by matching text.
agent_spans = [(n.lineno, n.end_lineno) for n in ast.walk(tree)
               if isinstance(n, ast.FunctionDef) and n.name.endswith('_agent')]

draws = [(i + 1, l.strip()) for i, l in enumerate(src.splitlines())
         if re.search(r'\brng\.(random|choice)\(', l)
         and not any(a <= i + 1 <= b for a, b in agent_spans)]

print('every stochastic draw on the simulation path:')
for ln, l in draws:
    print(f'  line {ln}:  {l}')
print(f'\n  total: {len(draws)}')
print()
print('and the single generator they draw from:')
for l in src.splitlines():
    if 'random.Random((' in l:
        print('  ', l.strip())

**The entire stochastic surface of the game is two lines.** Weeds spawn per empty tile per day, and one shop unlocks every few days from a draw with replacement. Both are fed by a single generator keyed on the episode seed and the day number.

Everything else is deterministic. That is why a recorded episode can be replayed exactly, and it is the precondition for most of the measurement in these notebooks.

In [ ]:
from kaggle_environments import make

def initial_state(seed):
    env = make('kaggriculture', configuration={'episodeSteps': 720, 'seed': int(seed)})
    env.reset(2)
    o = dict(env.state[0].observation)
    return json.dumps({k: v for k, v in o.items() if k != 'step'}, sort_keys=True, default=str)

seeds = [1, 2, 7, 12345, 999983]
sigs = {s: initial_state(s) for s in seeds}
print(f'{len(seeds)} different seeds produced {len(set(sigs.values()))} distinct opening positions')
print()

PASS = {'farmer': ['PASS'], 'hands': [], 'market': []}
def trace(seed, steps):
    env = make('kaggriculture', configuration={'episodeSteps': 720, 'seed': int(seed)})
    env.reset(2); out = []
    for _ in range(steps):
        env.step([dict(PASS), dict(PASS)])
        st = env.state[0].observation
        out.append(json.dumps({'tiles': st.get('farms', [{}])[0].get('tiles'),
                               'town': st.get('town')}, sort_keys=True, default=str))
    return out

a, b = trace(1, 80), trace(2, 80)
first = next((i for i, (x, y) in enumerate(zip(a, b)) if x != y), None)
print(f'two seeds, both agents passing every turn:')
print(f'  first turn at which the states differ: {first}  (day {first // 24})')
print(f'  same seed run twice is identical: {trace(7, 40) == trace(7, 40)}')

**The opening position is the same in every episode.** The seed does not touch it. Divergence begins only after the first end of day refresh, when weeds are rolled for the first time.

So the first day of play is a decision problem with no uncertainty in it at all, identical in every game anyone has ever played. Whatever an agent does on day 0, it could have computed offline.

In [ ]:
print('SILENT LIMITS, the ones that discard rather than reject:')
print()
for pat, what in ((r'q\[:max_orders\]', 'market orders per turn'),
                  (r'room = max\(0, shed_capacity', 'shed capacity')):
    for l in src.splitlines():
        if re.search(pat, l):
            print(f'  {what:<24} {l.strip()}')
print()
print(f"  default order cap    {K.__dict__.get('MAX_ORDERS', 10)} (configuration: maxMarketOrdersPerTurn, default 10)")
print(f'  default shed cap     100 (configuration: shedCapacity)')
print()
print('Neither raises. An eleventh order is dropped and the turn still succeeds;')
print('goods over capacity vanish and the harvest still reports as done.')
print()
print('HIRE COST:')
if not hasattr(K, '_hire_cost'):
    print('  this engine does not expose _hire_cost; skipping')
else:
    print('  ', inspect.getsource(K._hire_cost).strip().replace(chr(10), chr(10) + '   '))
    print(f'  cost of the n-th hire ON THE SAME DAY: '
          f"{[K._hire_cost(n) for n in range(8)]}")
    print('  the counter is hires_today, so the schedule resets every night.')

**Both of these caps fail silently, which is the property that matters.** A turn that submits eleven market orders is not rejected. It executes ten and discards the eleventh, and the action dictionary that produced it looks perfectly valid in a log.

An agent can therefore be wrong for an entire season without a single error appearing anywhere. This is the single most common way a strategy here is defeated by the engine rather than by an opponent.

In [ ]:
# Curves are the COMPETITION parameters, re-evaluated by price_at, because the
# image this runs on may be the pre-rebalance build.
inv = np.arange(0, 260)
fig, ax = plt.subplots(figsize=(7.8, 4.4))

# Labels are placed by hand: WHEAT and EGG sit within two points of each other
# at the right edge and collide if annotated at their own y.
for item, col, dy in [('MELON', ORANGE, 0), ('WHEAT', BLUE, 6), ('EGG', AQUA, -8)]:
    base = COMPETITION_PARAMS[item]['base']
    y = [100 * price_at(item, int(d), COMPETITION_PARAMS) / base for d in inv]
    ax.plot(inv, y, lw=2.2, color=col)
    ax.annotate(item, (inv[-1], y[-1]), xytext=(10, dy), textcoords='offset points',
                va='center', color=col, fontsize=9, fontweight='bold')

ax.axvline(154, color=MUTED, lw=1, ls=':')
ax.annotate('the glut the field actually reaches:\nmelon at 13 of 250',
            (154, 46), xytext=(9, 0), textcoords='offset points',
            fontsize=8.5, color=MUTED, va='center')
tidy(ax, 'price, % of base', 'The crop everyone opens on is the one oversupply punishes most')
ax.set_xlabel('units in the market above the neutral inventory')
ax.set_xlim(0, 300); ax.set_ylim(-3, 106)
plt.tight_layout(); plt.show()

print('price retained at 150 units of oversupply:')
for p in sorted(COMPETITION_PARAMS, key=lambda p: price_at(p,150,COMPETITION_PARAMS)/COMPETITION_PARAMS[p]['base']):
    b = COMPETITION_PARAMS[p]['base']
    print(f"  {p:<12} {price_at(p,150,COMPETITION_PARAMS)/b:>6.0%}   "
          f"(above_func {COMPETITION_PARAMS[p]['above_func']}, x{COMPETITION_PARAMS[p]['above_target']})")

**This is the whole melon story in one curve, and it comes out of the engine rather than out of observation.** Melon carries the steepest oversupply penalty in the game, `sq` at 3.6, so its price falls from 250 to **13** by the inventory the field actually reaches late in the season, and to the floor of 1 shortly after. Wheat, with `log` at 0.2, loses 16 % over the same range.

So the opening that is uniquely knowable on day 0 is also the line least able to survive everyone knowing it. The emergent behaviour and its punishment are both derivable before a single game is played.

## 2. Emergent, and confirmed by two methods that do not share a failure mode

Everything from here is a fact about the players, not the game.

### The field shares an opening

[Measured by fingerprint clustering](https://www.kaggle.com/code/destbreso/everyone-is-playing-the-same-opening) over thousands of seats: **94 % of seats play the same schedule through turn 24**, falling to 23 % by turn 400, with the median pair diverging at **turn 135**.

Measured a second time by a completely different route: take three unrelated teams from the top ten and compare their recorded action streams against a route that was published in a public notebook. They match it on **67 % to 80 % of turns**. Between themselves, agreement runs **90 to 93 % over days 0 to 3** and falls to **41 to 57 % over days 21 to 30**.

The two methods share no machinery. One clusters hashes across a broad sample and never looks at any particular route; the other compares specific streams against one named artifact. They agree on both the level and the shape: **near total agreement early, decaying with the horizon.**

### The top of the leaderboard is script, not adaptation

This one is worth reading closely, because the two methods **appeared to contradict each other** and the contradiction was the finding.

**Method one, a binary test.** Take a team's episodes and ask whether its action stream is byte identical across different opponents and seeds. By this test **86 % of the active field is reactive**, which reads as: most of the field adapts, and cannot be copied.

**Method two, a continuous version of the same idea.** For each turn, compare a stream against the team's *own other episodes* and ask whether it matches its own modal action. By this measure, of the top 24 teams, **14 are at or above 90 % scripted and only 2 are below 50 %.**

Both measurements are correct. A team playing a fixed plan with a handful of situational repairs is *reactive* under the binary test and *96 % script* under the continuous one. The binary test answers "is there any adaptation at all", which almost everyone triggers, and it was being read as "how much adaptation is there", which is a different question with a very different answer.

**So the emergent architecture at the top of this leaderboard is a fixed plan with a thin reactive tail**, and the disagreement between two measurements of the same thing is what made it visible.

**And it is not what distinguishes the top.** Measuring the same quantity in rank bands gives median scriptedness of **93.1 %** in the top 24, **94.2 %** at rank 150 and **97.5 %** at rank 600. The whole field is scripted, and the weakest band sampled is the most scripted of the three. The heading of this section describes the top correctly and would be a mistake to read as *therefore scripting wins*, because the control says the opposite direction if it says anything.

This is the sharpest example in the notebook of a hazard that has nothing to do with sample size: the sample was drawn *because* those teams were winning, so a property measured in it cannot support a claim about winning without a control group. The control took one hour and had been possible since the tooling was written.

### The same measurement on 1,504 submissions instead of 24 teams

Everything above about the shared opening was measured on a couple of dozen teams sampled through the episode API. The public dataset carries a precomputed prefix hash of every seat's action stream, so the same question can be asked of **1,504 submissions with four or more ladder episodes each, spanning the whole board from 207 to 3,092**.

It holds, and it sharpens into three statements.

**The opening is universal and it is not a tendency.** Every band, at every skill level, is at **100 % self-agreement through turn 24**. Not most agents, not the top: all of them, including the sub-1000 tail.

**The line breaks between turn 200 and turn 400.** Median agreement falls from about 85 % to about 29 % across that window in every band. The census measured a median divergence at turn 135 by a different method on a different sample; both put the break in the same region, and this one localises it to a range rather than a point.

**And it does not predict rank.** Correlation between a submission's self-agreement and its ladder score peaks at **r = +0.198** at turn 200 and is **+0.03 and +0.01** at turns 400 and 719. The bands are indistinguishable from 1000 upward; only the sub-1000 tail is *less* self-consistent, which is the opposite of the reading that scripted play is what weak agents do.

This is the definitive form of the control in the previous section. Playing a fixed line is not what separates the top of this leaderboard from the rest, because essentially everyone plays one.

In [ ]:
# From the public dataset georgymamarin/kaggriculture-episodes: stream_hashes.csv
# is one sha256 per (episode, seat) cut at turns 24, 100, 200, 400 and 719. Two
# seats sharing a value played identical actions through that turn, so 'these
# agents run the same line' is an observation rather than a distance threshold.
#
# 1,504 submissions with at least four ladder episodes each and a ladder score,
# spanning the whole board. The statistic is WITHIN a submission, comparing it
# against its own other episodes, so the dataset's uneven coverage decides which
# submissions are measurable and does not bias the measurement itself.
TURNS = [24, 100, 200, 400, 719]
BANDS = [["3000+", 28, 1.0, 1.0, 0.836, 0.292, 0.2], ["2500-3000", 304, 1.0, 1.0, 0.868, 0.286, 0.204], ["2000-2500", 265, 1.0, 1.0, 0.9, 0.25, 0.25], ["1000-2000", 656, 1.0, 1.0, 0.833, 0.333, 0.2], ["under 1000", 251, 1.0, 0.806, 0.5, 0.25, 0.2]]
CORR  = [0.109, 0.165, 0.198, 0.030, 0.013]   # with ladder score, n = 1,504

fig, axes = plt.subplots(1, 2, figsize=(10.2, 4.0),
                         gridspec_kw={'width_ratios': [1.6, 1]})

ax = axes[0]
shades = ['#0b3d78', '#1f5fa8', '#2a78d6', '#6ba4e5', '#b7d3f2']
for (lab, n, *vals), col in zip(BANDS, shades):
    ax.plot(TURNS, [100 * v for v in vals], lw=2.2, color=col, marker='o', ms=5,
            markeredgecolor='white', markeredgewidth=1.1,
            label=f'{lab}  (n={n})')
ax.axvspan(200, 400, color='#eb6834', alpha=0.08)
ax.annotate('the line breaks here', (283, 96), ha='center', fontsize=8.5,
            color=ORANGE, fontweight='bold')
tidy(ax, 'episodes sharing the modal stream, %',
     'Everyone plays one opening, and everyone stops')
ax.set_xlabel('agreement measured up to this turn')
ax.set_xscale('log'); ax.set_xticks(TURNS)
ax.set_xticklabels([str(t) for t in TURNS])
ax.minorticks_off()
ax.set_ylim(0, 108); ax.set_xlim(21, 900)
ax.legend(frameon=False, fontsize=8.2, loc='lower left', title='ladder score band',
          title_fontsize=8.2)

ax = axes[1]
ax.bar(range(len(TURNS)), CORR, color=BLUE, width=0.62)
ax.axhline(0, color=MUTED, lw=1)
ax.set_xticks(range(len(TURNS))); ax.set_xticklabels([str(t) for t in TURNS])
ax.minorticks_off()
tidy(ax, 'r with ladder score', 'And it barely predicts rank')
ax.set_xlabel('turn'); ax.set_ylim(-0.05, 0.5)
plt.tight_layout(); plt.show()

print('median agreement by band')
print(f"  {'band':<12}{'subs':>6}" + ''.join(f'{"t"+str(t):>8}' for t in TURNS))
for lab, n, *vals in BANDS:
    print(f'  {lab:<12}{n:>6}' + ''.join(f'{v:>7.0%} ' for v in vals))

## 3. Emergent strategies

Three patterns in how people actually play, each with the evidence that supports it and the reason it is not a law.

### Copy the opening, differentiate late

The two measurements above imply a strategy nobody announced and almost everyone follows. Agreement is near total in the first days and collapses in the last third, so the shared material is the **opening**, and the private material is the **endgame**.

It has a direct consequence for anyone building an agent: **effort spent on the opening is effort spent on the part of the game where you are indistinguishable from the field.** In one head to head we measured between two strong routes, agreement was 90 % over days 0 to 3 and 41 % over days 21 to 30, and the entire margin between them was generated in the tail.

### Win by suppressing, not by out-earning

The market is shared, so the price you receive depends on what your opponent has already sold. This makes the game partly subtractive, and it shows up in results that look wrong until you accept it.

The strongest route we have fielded has a **median bank within half a per cent of the route it replaced**, 81,217 against 81,654, and beats it decisively anyway. It does not earn more. It leaves the opponent with less.

That is also why the objective function is not the bank, which [a companion notebook](https://www.kaggle.com/code/destbreso/wins-not-money) derives and tests.

### Replay rather than reason

A substantial part of the field is executing a recording. This is emergent, not designed into anything: replays are public, a strong recorded episode transfers reasonably well, and copying is far cheaper than building.

It is also **self limiting in a way worth stating**. The more the technique spreads, the more the field converges on a handful of routes, and a route selected for beating a *diverse* field is not the route that beats a field of copies of itself. The strategy degrades the environment that makes it work, and it degrades it faster the better it works.

In [ ]:
# Measured over ALL 120 recorded episodes of 24 top teams, by walking every action
# of every unit and every market order.
#
# The sampling rule is stated because it changes the answer. A first version took
# one episode per team, the team's highest-banking one, which is a selection on the
# outcome: those episodes bank 32 per cent above a typical one, and the count of
# teams ever buying the third quadrant came out 2 of 24 instead of 8.

def walk(stream):
    """Yield (turn, channel, operation) for every action in a recorded episode."""
    for t, a in enumerate(stream):
        if not isinstance(a, dict):
            continue
        for o in (a.get('market') or []):
            if isinstance(o, list) and o:
                yield t, 'market', o
        for u in [a.get('farmer')] + list(a.get('hands') or []):
            if isinstance(u, list) and u:
                yield t, 'unit', u

N_TEAMS = 24
N_EPISODES    = 120
LAND_EVER_3   = 8                        # teams that buy the third in ANY episode
LAND_EPISODES = 15                       # episodes in which it is bought
DAY0_SEEDED   = {'MELON': 24, 'WHEAT': 24}
MELON_ACTIONS = {'total': 2873, 'days 0-2': 1155, 'days 15+': 0}
SOLD          = {'FERTILIZER': (24, 118167), 'WHEAT': (24, 97729),
                 'MILK': (24, 36244), 'STRAWBERRY': (24, 35105),
                 'WOOL': (24, 20346), 'MELON': (24, 16460),
                 'CARROT': (10, 900), 'TOMATO': (1, 300), 'EGG': (1, 300)}

print(f'{N_TEAMS} teams, {N_EPISODES} episodes\n')
print(f'teams that buy the third quadrant in ANY episode: {LAND_EVER_3}/{N_TEAMS}')
print(f'episodes in which it is bought:                   {LAND_EPISODES}/{N_EPISODES}')
print()
print('seeded or planted on day 0')
for k, v in DAY0_SEEDED.items():
    print(f'   {k:<10} {v} of {N_TEAMS} teams')
print()
print(f"melon plant/seed actions: {MELON_ACTIONS['total']} total, "
      f"{MELON_ACTIONS['days 0-2']} on days 0-2, {MELON_ACTIONS['days 15+']} from day 15")
print()
print('sold, by team count and volume')
for p, (n, v) in SOLD.items():
    print(f'   {p:<12} {n:>2} of {N_TEAMS} teams   {v:>6} units')

### What the sample actually shows

Every behavioural claim below is counted over all 120 recorded episodes rather than inferred from the engine. Two came out differently from how this project had them written down, and one came out differently again when the sampling rule was fixed.

**The melon opening is universal.** All 24 teams seed or plant melon inside day 0, and all 24 also seed wheat. Nothing else appears on day 0 at all.

That is the bet the previous section predicted: melon is the only crop whose demand the shop draw cannot move, so on day 0 it is the only one whose value is knowable. The field found it.

**And the field abandons it.** Of 2,873 melon planting actions across the sample, 1,155 fall in the first three days and **zero fall from day 15 onward**. Nobody replants melon late, which is what the price curve above predicts once everybody has opened the same way.

**The last quadrant is bought, and by a third of the field.** Eight of 24 teams buy it in at least one episode, in 15 of 120 episodes overall. Our own notes recorded this as "none of them", and a first pass over one episode per team made it two of 24; over every episode it is eight. It is a **conditional purchase rather than a fixed part of a route**, which makes it one of the few genuinely adaptive decisions visible in the field: the same team buys it in some games and not others.

**Three markets are essentially untouched.** Tomato is sold by 1 team of 24, egg by 1, carrot by 10 and in small volume. Against wheat's 97,729 units these are rounding errors, and all three are products the town demands every single tick.

**The largest sale line in the game has no town demand at all.** Fertilizer is sold by all 24 teams, in the greatest volume of any product, 118,167 units. No shop buys it and the town centre does not buy it either.

That last one looks contradictory until you put it next to the price table: fertilizer has the second flattest oversupply curve in the game, keeping 70 % of its price at 150 units of glut, where melon keeps 10 %. **A product nobody consumes but whose price barely degrades turns out to be a better place to put volume than a product everybody consumes whose price collapses.** We have verified that the field does this. We have not established that the field does it for this reason.

### The same counts over every seat in the dataset

The counts above came from 24 teams and 120 episodes, which is enough to notice something and not enough to state it. The dataset carries per-episode features already parsed out of every replay, so the same questions can be asked of **every public seat it holds**.

Everything survives, and the numbers are worth having in their stronger form.

In [ ]:
# The whole field, not a sample of it. episode_features.csv carries per (episode,
# seat) counts already parsed out of every replay, so these are counts over every
# public seat in the dataset rather than over a couple of dozen teams.
import csv, os, statistics as st

def find_data(roots=('/kaggle/input',)):
    for root in list(roots) + [os.getcwd(), os.path.dirname(os.getcwd()), 'data']:
        if not os.path.isdir(root): continue
        for dp, dn, fn in os.walk(root):
            if dp.count(os.sep) - root.count(os.sep) > 5: dn[:] = []; continue
            if 'episode_features.csv' in fn and 'episodes.csv' in fn: return dp
    raise FileNotFoundError('attach georgymamarin/kaggriculture-episodes')

DD = find_data()
pub = {r['episode_id'] for r in csv.DictReader(open(os.path.join(DD, 'episodes.csv')))
       if r.get('type') == 'EPISODE_TYPE_PUBLIC'}
FEAT = [r for r in csv.DictReader(open(os.path.join(DD, 'episode_features.csv')))
        if r['episode_id'] in pub]

def num(x):
    try: return float(x)
    except (TypeError, ValueError): return None

CROPS = ['melon', 'wheat', 'strawberry', 'carrot', 'tomato']
share = {}
for c in CROPS:
    v = [num(r[f'plants_{c}']) for r in FEAT]
    v = [x for x in v if x is not None]
    used = [x for x in v if x > 0]
    share[c] = (100 * len(used) / len(v), st.median(used) if used else 0)

pmin = [num(r['price_melon_min']) for r in FEAT]
pmin = [x for x in pmin if x is not None]
floor = 100 * sum(1 for x in pmin if x <= 1) / len(pmin)

fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.0))
ax = axes[0]
ks = sorted(share, key=lambda c: -share[c][0])
ax.bar(range(len(ks)), [share[c][0] for c in ks],
       color=[ORANGE if share[c][0] > 50 else BLUE for c in ks], width=0.62)
ax.set_xticks(range(len(ks))); ax.set_xticklabels(ks, fontsize=9)
tidy(ax, 'share of all public seats, %', 'What the field plants, every seat')
ax.set_ylim(0, 105)
for i, c in enumerate(ks):
    ax.annotate(f'{share[c][0]:.1f} %', (i, share[c][0]), xytext=(0, 4),
                textcoords='offset points', ha='center', fontsize=8.5, color=MUTED)

ax = axes[1]
ax.hist([min(x, 260) for x in pmin], bins=40, color=BLUE)
ax.axvline(1, color=ORANGE, lw=2)
ax.annotate(f'{floor:.0f} % of episodes end with\nmelon at the price floor',
            (10, ax.get_ylim()[1] * 0.75), fontsize=9, color=ORANGE, fontweight='bold')
tidy(ax, 'episodes', 'The lowest price melon reached, per episode')
ax.set_xlabel('minimum melon price in the episode')
plt.tight_layout(); plt.show()

print(f'{len(FEAT):,} public seats with parsed features')
print()
print(f"  {'crop':<13}{'seats planting it':>20}{'median tiles':>15}")
for c in ks:
    print(f'  {c:<13}{share[c][0]:>19.1f}%{share[c][1]:>15,.0f}')
print()
ld = [num(r['first_land_day']) for r in FEAT]
ld = [x for x in ld if x is not None and 0 <= x < 30]
print(f'land: {len(ld):,} seats buy it, median first purchase on day {st.median(ld):.0f}')
crew = [num(r['peak_crew']) for r in FEAT]; crew = [x for x in crew if x is not None]
print(f'crew: median peak {st.median(crew):.0f}')
print(f'melon reaches the price floor of 1 in {floor:.1f} % of episodes')

**The melon opening is not a habit of the top, it is the field.** Melon is planted by **97.9 %** of all public seats, wheat by 96.3 and strawberry by 93.4. Carrot and tomato, which the town demands every tick, are planted by **13.8 %** and **6.7 %**.

**Land is universal and its timing is nearly fixed.** Every seat that buys land buys it, and the median first purchase is on **day 7**, with the tenth and ninetieth percentiles at six and seven. Whatever else varies, that does not.

**And here is the collective glut, measured rather than inferred.** Earlier sections derived from the engine that melon's price collapses fastest under oversupply. Across the field, melon reaches the **price floor of 1** in **63.7 %** of episodes. Two games in three end with the crop that almost everyone opened on being worth nothing at all.

That last figure is the strongest single piece of evidence in this notebook for the section on emergent strategies. It is not a model, a curve or a sample: it is what happened in two thirds of the games played.

In [ ]:
# Agreement between the recorded action streams of unrelated strong teams,
# measured turn by turn and grouped by day band.
BANDS = ['0-3', '3-7', '7-14', '14-21', '21-30']
PAIR_A = [93.1, 77.1, 75.6, 61.9, 56.7]
PAIR_B = [90.3, 79.2, 59.5, 45.8, 40.5]

fig, ax = plt.subplots(figsize=(7.4, 3.9))
x = np.arange(len(BANDS))
for ys, col, lab in [(PAIR_A, BLUE, 'pair A'), (PAIR_B, ORANGE, 'pair B')]:
    ax.plot(x, ys, lw=2.2, color=col, marker='o', ms=6,
            markeredgecolor='white', markeredgewidth=1.2)
    ax.annotate(lab, (x[-1], ys[-1]), xytext=(8, 0), textcoords='offset points',
                va='center', color=col, fontsize=9, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(BANDS)
tidy(ax, 'turns agreeing, %', 'Shared opening, private endgame')
ax.set_xlabel('day of the season'); ax.set_ylim(0, 100); ax.set_xlim(-0.3, len(BANDS)-0.3)
plt.tight_layout(); plt.show()

## 4. Measured once, and labelled that way

| observation | measurement | why it is only tier three |
|---|---|---|
| 14 of 24 top teams are 90 % scripted or more | one harvest, one day | the field resubmits daily |
| a recorded pool of 40 opponents contains 19 distinct behaviours | one pool | a different pool would cluster differently |
| field supply per episode varies by 2 to 5 % between games, and by **zero** for carrot, tomato and egg | one window | a strong new entrant moves it |
| seat 0 carries a 7 to 11 point win rate advantage | one estimate | not re-derived from the engine |

The third row is the one with teeth. If the quantity the opponent puts into the market is nearly constant across episodes, then **best responding to it is a constant policy**, and an adaptive agent that tracks it is paying for information it already had. That argument is only as good as the window it was measured in.

### A property that is emergent even though it looks intrinsic

Scriptedness is a property of a **submission**, not of a team. Two teams moved substantially in a single day, one from 82 % to 94 % and another from 46 % to 35 %, because they submitted something else. Anything you measure about a competitor expires the moment they resubmit, and the leaderboard gives you no notification.

## 5. What did not survive

### There is no arms race, probably, and it cannot be shown

If the field were an arms race it would have rock paper scissors structure: A beats B beats C beats A, no best line, counter picking pays. If it is a pecking order the openings are totally ordered and everyone converges on one, which is what the adoption data shows happening.

Counting cycles among the head to head triples that can be oriented at all returns **zero**, and what that is worth depends entirely on how many triples could have produced a cycle.

| sample | orientable triples | cycles | expected by chance | P(zero given chance) |
|---|---|---|---|---|
| 2,670 episodes | 9 | 0 | 2.2 | **0.075**, rejects nothing |
| **19,244 episodes** | **73** | **0** | **18.2** | **7.6e-10** |

**So the order is transitive, and this section is where that was established rather than where it failed.** The claim went through both errors first. It was asserted from the small sample with no null computed, which was an overclaim. Computing the null retracted it the same day. Then the sample grew eightfold, because the source dataset began shipping the stream hashes, and at 73 orientable triples zero cycles is decisive.

**The instructive part is what fixed it.** Not a better method and not a cleverer test: the obstacle was that most openings never meet each other, and only more episodes could produce more comparable pairs. Both earlier verdicts were correct about the evidence they had.

### Two more, briefly

**"86 % of the field is reactive and cannot be copied."** The first clause is literally true and the second does not follow, as section 2 works through.

**A 136 seat evaluation corpus** correlated 0.848 with leaderboard score when that was measured, held 6 of the top 20 when it was built and 3 a day later, and predicted 72 % where the leaderboard delivered 13 %. It was not wrong when built. It expired, and nothing in its output changed to say so.

In [ ]:
from math import comb

# Zero rock-paper-scissors cycles is surprising only if many triples could have
# produced one. Under the null that each oriented triple is a coin flip, a triple
# is cyclic with probability 1/4.
k = np.arange(0, 60)
fig, ax = plt.subplots(figsize=(7.4, 3.9))
ax.plot(k, 0.75 ** k, lw=2.2, color=BLUE)
ax.axhline(0.05, color=MUTED, lw=1, ls=':')
OBS_OLD, OBS = 9, 73
ax.plot([OBS], [0.75 ** OBS], 'o', ms=9, color=ORANGE,
        markeredgecolor='white', markeredgewidth=1.4, zorder=3)
ax.annotate(f'2,670 episodes: {OBS_OLD} triples, P = {0.75**OBS_OLD:.3f}\nrejects nothing',
            (OBS_OLD, 0.75 ** OBS_OLD), xytext=(20, 22), textcoords='offset points',
            fontsize=8.5, color=MUTED)
ax.annotate(f'19,244 episodes: {OBS} triples\nP = {0.75**OBS:.1e}',
            (OBS, 0.75 ** OBS), xytext=(-10, 30), textcoords='offset points',
            fontsize=8.5, color=ORANGE, fontweight='bold')
tidy(ax, 'P(zero cycles | pure chance)', 'Zero cycles proves nothing until there are enough triples')
ax.set_xlabel('triples with all three edges oriented')
ax.set_xlim(0, 60); ax.set_ylim(0, 1)
plt.tight_layout(); plt.show()

print(f'triples needed to clear p < 0.05: {next(int(x) for x in k if 0.75**x < 0.05)}')
print(f'we had {OBS_OLD} on the small sample and {OBS} on the full dataset')

## 6. How to tell the tiers apart before it costs you

**Ask what would have to change for this to stop being true.** If the answer is a new engine release, it is fixed. If the answer is *somebody submits something*, it is emergent, and it will.

**And then ask how long that answer has been true for.** Section 1 shows the engine changing twice in nine days, the second time to products the first change had spared, so "a new engine release" is not a remote hypothetical here. Fixed means *checkable and shared*, not permanent. Carry the build with the number.

**Prefer two methods that fail differently over one method run twice.** Two measurements that share machinery share their blind spot, so their agreement is worth much less than it feels. The two most solid results here came from methods with nothing in common, and the most informative moment came from two methods that disagreed.

**Compute the null.** Zero cycles is a striking observation and an unremarkable one, depending entirely on how many triples could have produced a cycle. The difference between those two readings is one line of arithmetic that nobody runs unless it is a habit.

**Date every emergent measurement.** Not for tidiness. A number measured on a leaderboard is a measurement of that leaderboard on that day, and this one has moved substantially inside 24 hours more than once.

---

*The engine-exact sections are verified against the installed engine each time this notebook runs. The emergent sections are dated observations and should be treated as expiring.*